In [4]:
# Folder sumber
dataset_source_all_class = "../dataset_collection/roboflow/object_detection/object_detection_all_class"
dataset_source_healthy_rust = "../dataset_collection/roboflow/object_detection/object_detection_few_shot_healthy_rust" 
# Folder tujuan
dataset_destination_all_class = "../dataset_collection/roboflow/k-shot/object_detection_all_class"
dataset_destination_healthy_rust = "../dataset_collection/roboflow/k-shot/object_detection_healthy_rust"

## Split all class

In [2]:
import os
import shutil
import random
import yaml
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

In [8]:
# ==========================================
# KONFIGURASI
# ==========================================
SOURCE_DIR = dataset_source_all_class        # Path folder dataset asli
DEST_DIR = dataset_destination_all_class   # Path folder tujuan
K_VALUES = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
RANDOM_SEED = 42

# Ekstensi gambar yang didukung
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def load_yaml_info(yaml_path):
    """Membaca file data.yaml."""
    if not os.path.exists(yaml_path):
        raise FileNotFoundError(f"File data.yaml tidak ditemukan di {yaml_path}")
    
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    if 'names' not in data:
        raise ValueError("File data.yaml harus memiliki key 'names'")
    
    names = data['names']
    # Normalisasi ke list jika formatnya dict
    if isinstance(names, dict):
        sorted_keys = sorted(names.keys())
        class_list = [names[k] for k in sorted_keys]
    else:
        class_list = names
        
    return data, class_list

def normalize_string(s):
    """Mengubah string agar konsisten (misal: 'frog-eye' jadi 'frog_eye')."""
    return s.lower().replace('-', '_').replace(' ', '_')

def get_class_from_filename(filename, class_names):
    """
    Menentukan Class ID berdasarkan nama file.
    """
    fname_norm = normalize_string(filename)
    found_classes = []
    
    for idx, name in enumerate(class_names):
        # Normalisasi nama kelas dari YAML
        clean_name = normalize_string(name)
        if clean_name in fname_norm:
            found_classes.append(idx)
            
    # Jika ada match ganda, ambil string terpanjang (paling spesifik)
    if len(found_classes) > 1:
        found_classes.sort(key=lambda i: len(class_names[i]), reverse=True)
        return found_classes[0]
    
    if len(found_classes) == 1:
        return found_classes[0]
        
    return None

def copy_static_folders(source_base, dest_base):
    """
    Menyalin folder valid dan test ke root destination satu kali saja.
    """
    print("\n[*] Menyalin folder valid dan test (One-time copy)...")
    for split in ['valid', 'test']:
        src_path = os.path.join(source_base, split)
        dst_path = os.path.join(dest_base, split)
        
        if os.path.exists(src_path):
            if os.path.exists(dst_path):
                print(f"    - Folder {split} sudah ada di tujuan, skip copy.")
            else:
                shutil.copytree(src_path, dst_path)
                print(f"    - Berhasil menyalin {split}")
        else:
            print(f"    - [WARNING] Folder {split} tidak ditemukan di source.")

def get_train_pairs(source_base, class_names):
    """
    Mengindeks gambar training berdasarkan nama file.
    """
    train_img_dir = os.path.join(source_base, "train", "images")
    train_lbl_dir = os.path.join(source_base, "train", "labels")
    
    if not os.path.exists(train_img_dir):
        raise FileNotFoundError("Folder train/images tidak ditemukan.")

    class_to_images = defaultdict(list)
    pairs_map = {} 
    
    img_files = [f for f in os.listdir(train_img_dir) if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS]
    print(f"\n[*] Mengindeks {len(img_files)} gambar training...")

    for img_file in tqdm(img_files, desc="Indexing Train Data"):
        stem = os.path.splitext(img_file)[0]
        lbl_file = stem + ".txt"
        
        img_path = os.path.join(train_img_dir, img_file)
        lbl_path = os.path.join(train_lbl_dir, lbl_file)
        
        # Cek label
        if not os.path.exists(lbl_path):
            continue 
            
        # Cek kelas dari nama file
        class_id = get_class_from_filename(img_file, class_names)
        if class_id is None:
            continue

        pairs_map[img_file] = {'img': img_path, 'lbl': lbl_path}
        class_to_images[class_id].append(img_file)
        
    return class_to_images, pairs_map

def create_k_shot_subset(k, class_to_images, pairs_map, class_names, yaml_data, output_base):
    """
    Membuat folder {k}-shot berisi images/labels dan file {k}-shot.yaml
    """
    k_shot_dir_name = f"{k}-shot"
    k_shot_full_path = os.path.join(output_base, k_shot_dir_name)
    
    out_img_dir = os.path.join(k_shot_full_path, "images")
    out_lbl_dir = os.path.join(k_shot_full_path, "labels")
    
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    
    # --- SAMPLING ---
    selected_files = set()
    for class_id, class_name in enumerate(class_names):
        available = class_to_images.get(class_id, [])
        if len(available) < k:
            raise ValueError(f"Kelas '{class_name}' hanya punya {len(available)} gambar, diminta {k}.")
        
        sampled = random.sample(available, k)
        selected_files.update(sampled)
        
    # --- COPY TRAIN FILES ---
    for img_file in selected_files:
        p = pairs_map[img_file]
        shutil.copy2(p['img'], os.path.join(out_img_dir, img_file))
        shutil.copy2(p['lbl'], os.path.join(out_lbl_dir, os.path.basename(p['lbl'])))
        
    # --- GENERATE YAML ---
    new_yaml = yaml_data.copy()
    
    # Ubah 'train' path. Menggunakan ./ agar relatif terhadap lokasi file YAML baru di root dest.
    # Contoh: ./5-shot/images
    new_yaml['train'] = f"../{k_shot_dir_name}/images"
    
    # Biarkan 'val' dan 'test' default (biasanya val: valid/images).
    # Karena folder valid ada di root destination (sejajar dengan file yaml ini),
    # maka path default biasanya tetap valid.
    
    yaml_out = os.path.join(output_base, f"{k}-shot.yaml")
    with open(yaml_out, 'w') as f:
        yaml.dump(new_yaml, f, sort_keys=False)
        
    return k_shot_full_path

def analyze_stats(k, k_shot_path, class_names):
    """Hitung distribusi file di folder hasil."""
    img_dir = os.path.join(k_shot_path, "images")
    img_files = os.listdir(img_dir)
    
    counts = defaultdict(int)
    for f in img_files:
        cid = get_class_from_filename(f, class_names)
        if cid is not None:
            counts[cid] += 1
            
    stats = []
    for idx, name in enumerate(class_names):
        stats.append({
            "k": k,
            "class_name": name,
            "num_images": counts[idx]
        })
    return stats

def main():
    random.seed(RANDOM_SEED)
    
    # Bersihkan output jika perlu (Hati-hati, ini menghapus folder tujuan!)
    if os.path.exists(DEST_DIR):
        print(f"[*] Membersihkan folder tujuan: {DEST_DIR}")
        shutil.rmtree(DEST_DIR)
    os.makedirs(DEST_DIR)
    
    print(f"[*] Source: {SOURCE_DIR}")
    print(f"[*] Dest  : {DEST_DIR}")

    # 1. Baca Config
    yaml_path = os.path.join(SOURCE_DIR, "data.yaml")
    yaml_data, class_names = load_yaml_info(yaml_path)
    print(f"[*] Kelas: {class_names}")

    # 2. Copy Valid & Test (Sekali saja ke Root)
    copy_static_folders(SOURCE_DIR, DEST_DIR)
    
    # 3. Indexing Data Train
    class_to_images, pairs_map = get_train_pairs(SOURCE_DIR, class_names)
    
    # Debug info
    print("[*] Jumlah gambar tersedia per kelas:")
    for idx, name in enumerate(class_names):
        print(f"    - {name}: {len(class_to_images[idx])}")

    # 4. Loop K-Shot
    all_stats = []
    
    for k in K_VALUES:
        print(f"\n--- Membuat {k}-Shot ---")
        try:
            k_path = create_k_shot_subset(k, class_to_images, pairs_map, class_names, yaml_data, DEST_DIR)
            stats = analyze_stats(k, k_path, class_names)
            all_stats.extend(stats)
            print(f"    -> Selesai. YAML: {k}-shot.yaml")
        except ValueError as e:
            print(f"    [FAILED] {e}")
            break # Stop jika data tidak cukup
        except Exception as e:
            print(f"    [ERROR] {e}")
            break

    # 5. Report
    if all_stats:
        csv_path = os.path.join(DEST_DIR, "kshot_class_distribution.csv")
        df = pd.DataFrame(all_stats)
        df.to_csv(csv_path, index=False)
        print(f"\n[*] Laporan CSV tersimpan: {csv_path}")

if __name__ == "__main__":
    main()

[*] Source: ../dataset_collection/roboflow/object_detection/object_detection_all_class
[*] Dest  : ../dataset_collection/roboflow/k-shot/object_detection_all_class
[*] Kelas: ['frog-eye-leaf-spot', 'healthy', 'rust']

[*] Menyalin folder valid dan test (One-time copy)...
    - Berhasil menyalin valid
    - Berhasil menyalin test

[*] Mengindeks 525 gambar training...


Indexing Train Data: 100%|██████████| 525/525 [00:00<00:00, 79231.78it/s]

[*] Jumlah gambar tersedia per kelas:
    - frog-eye-leaf-spot: 175
    - healthy: 175
    - rust: 175

--- Membuat 5-Shot ---
    -> Selesai. YAML: 5-shot.yaml

--- Membuat 10-Shot ---
    -> Selesai. YAML: 10-shot.yaml

--- Membuat 15-Shot ---
    -> Selesai. YAML: 15-shot.yaml

--- Membuat 20-Shot ---
    -> Selesai. YAML: 20-shot.yaml

--- Membuat 25-Shot ---


    -> Selesai. YAML: 25-shot.yaml

--- Membuat 30-Shot ---
    -> Selesai. YAML: 30-shot.yaml

--- Membuat 35-Shot ---
    -> Selesai. YAML: 35-shot.yaml

--- Membuat 40-Shot ---
    -> Selesai. YAML: 40-shot.yaml

--- Membuat 45-Shot ---
    -> Selesai. YAML: 45-shot.yaml

--- Membuat 50-Shot ---
    -> Selesai. YAML: 50-shot.yaml

[*] Laporan CSV tersimpan: ../dataset_collection/roboflow/k-shot/object_detection_all_class\kshot_class_distribution.csv


In [5]:
# ==========================================
# KONFIGURASI
# ==========================================
SOURCE_DIR = dataset_source_healthy_rust        # Path folder dataset asli
DEST_DIR = dataset_destination_healthy_rust  # Path folder tujuan
K_VALUES = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
RANDOM_SEED = 42

# Ekstensi gambar yang didukung
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def load_yaml_info(yaml_path):
    """Membaca file data.yaml."""
    if not os.path.exists(yaml_path):
        raise FileNotFoundError(f"File data.yaml tidak ditemukan di {yaml_path}")
    
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    if 'names' not in data:
        raise ValueError("File data.yaml harus memiliki key 'names'")
    
    names = data['names']
    # Normalisasi ke list jika formatnya dict
    if isinstance(names, dict):
        sorted_keys = sorted(names.keys())
        class_list = [names[k] for k in sorted_keys]
    else:
        class_list = names
        
    return data, class_list

def normalize_string(s):
    """Mengubah string agar konsisten (misal: 'frog-eye' jadi 'frog_eye')."""
    return s.lower().replace('-', '_').replace(' ', '_')

def get_class_from_filename(filename, class_names):
    """
    Menentukan Class ID berdasarkan nama file.
    """
    fname_norm = normalize_string(filename)
    found_classes = []
    
    for idx, name in enumerate(class_names):
        # Normalisasi nama kelas dari YAML
        clean_name = normalize_string(name)
        if clean_name in fname_norm:
            found_classes.append(idx)
            
    # Jika ada match ganda, ambil string terpanjang (paling spesifik)
    if len(found_classes) > 1:
        found_classes.sort(key=lambda i: len(class_names[i]), reverse=True)
        return found_classes[0]
    
    if len(found_classes) == 1:
        return found_classes[0]
        
    return None

def copy_static_folders(source_base, dest_base):
    """
    Menyalin folder valid dan test ke root destination satu kali saja.
    """
    print("\n[*] Menyalin folder valid dan test (One-time copy)...")
    for split in ['valid', 'test']:
        src_path = os.path.join(source_base, split)
        dst_path = os.path.join(dest_base, split)
        
        if os.path.exists(src_path):
            if os.path.exists(dst_path):
                print(f"    - Folder {split} sudah ada di tujuan, skip copy.")
            else:
                shutil.copytree(src_path, dst_path)
                print(f"    - Berhasil menyalin {split}")
        else:
            print(f"    - [WARNING] Folder {split} tidak ditemukan di source.")

def get_train_pairs(source_base, class_names):
    """
    Mengindeks gambar training berdasarkan nama file.
    """
    train_img_dir = os.path.join(source_base, "train", "images")
    train_lbl_dir = os.path.join(source_base, "train", "labels")
    
    if not os.path.exists(train_img_dir):
        raise FileNotFoundError("Folder train/images tidak ditemukan.")

    class_to_images = defaultdict(list)
    pairs_map = {} 
    
    img_files = [f for f in os.listdir(train_img_dir) if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS]
    print(f"\n[*] Mengindeks {len(img_files)} gambar training...")

    for img_file in tqdm(img_files, desc="Indexing Train Data"):
        stem = os.path.splitext(img_file)[0]
        lbl_file = stem + ".txt"
        
        img_path = os.path.join(train_img_dir, img_file)
        lbl_path = os.path.join(train_lbl_dir, lbl_file)
        
        # Cek label
        if not os.path.exists(lbl_path):
            continue 
            
        # Cek kelas dari nama file
        class_id = get_class_from_filename(img_file, class_names)
        if class_id is None:
            continue

        pairs_map[img_file] = {'img': img_path, 'lbl': lbl_path}
        class_to_images[class_id].append(img_file)
        
    return class_to_images, pairs_map

def create_k_shot_subset(k, class_to_images, pairs_map, class_names, yaml_data, output_base):
    """
    Membuat folder {k}-shot berisi images/labels dan file {k}-shot.yaml
    """
    k_shot_dir_name = f"{k}-shot"
    k_shot_full_path = os.path.join(output_base, k_shot_dir_name)
    
    out_img_dir = os.path.join(k_shot_full_path, "images")
    out_lbl_dir = os.path.join(k_shot_full_path, "labels")
    
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    
    # --- SAMPLING ---
    selected_files = set()
    for class_id, class_name in enumerate(class_names):
        available = class_to_images.get(class_id, [])
        if len(available) < k:
            raise ValueError(f"Kelas '{class_name}' hanya punya {len(available)} gambar, diminta {k}.")
        
        sampled = random.sample(available, k)
        selected_files.update(sampled)
        
    # --- COPY TRAIN FILES ---
    for img_file in selected_files:
        p = pairs_map[img_file]
        shutil.copy2(p['img'], os.path.join(out_img_dir, img_file))
        shutil.copy2(p['lbl'], os.path.join(out_lbl_dir, os.path.basename(p['lbl'])))
        
    # --- GENERATE YAML ---
    new_yaml = yaml_data.copy()
    
    # Ubah 'train' path. Menggunakan ./ agar relatif terhadap lokasi file YAML baru di root dest.
    # Contoh: ./5-shot/images
    new_yaml['train'] = f"../{k_shot_dir_name}/images"
    
    # Biarkan 'val' dan 'test' default (biasanya val: valid/images).
    # Karena folder valid ada di root destination (sejajar dengan file yaml ini),
    # maka path default biasanya tetap valid.
    
    yaml_out = os.path.join(output_base, f"{k}-shot.yaml")
    with open(yaml_out, 'w') as f:
        yaml.dump(new_yaml, f, sort_keys=False)
        
    return k_shot_full_path

def analyze_stats(k, k_shot_path, class_names):
    """Hitung distribusi file di folder hasil."""
    img_dir = os.path.join(k_shot_path, "images")
    img_files = os.listdir(img_dir)
    
    counts = defaultdict(int)
    for f in img_files:
        cid = get_class_from_filename(f, class_names)
        if cid is not None:
            counts[cid] += 1
            
    stats = []
    for idx, name in enumerate(class_names):
        stats.append({
            "k": k,
            "class_name": name,
            "num_images": counts[idx]
        })
    return stats

def main():
    random.seed(RANDOM_SEED)
    
    # Bersihkan output jika perlu (Hati-hati, ini menghapus folder tujuan!)
    if os.path.exists(DEST_DIR):
        print(f"[*] Membersihkan folder tujuan: {DEST_DIR}")
        shutil.rmtree(DEST_DIR)
    os.makedirs(DEST_DIR)
    
    print(f"[*] Source: {SOURCE_DIR}")
    print(f"[*] Dest  : {DEST_DIR}")

    # 1. Baca Config
    yaml_path = os.path.join(SOURCE_DIR, "data.yaml")
    yaml_data, class_names = load_yaml_info(yaml_path)
    print(f"[*] Kelas: {class_names}")

    # 2. Copy Valid & Test (Sekali saja ke Root)
    copy_static_folders(SOURCE_DIR, DEST_DIR)
    
    # 3. Indexing Data Train
    class_to_images, pairs_map = get_train_pairs(SOURCE_DIR, class_names)
    
    # Debug info
    print("[*] Jumlah gambar tersedia per kelas:")
    for idx, name in enumerate(class_names):
        print(f"    - {name}: {len(class_to_images[idx])}")

    # 4. Loop K-Shot
    all_stats = []
    
    for k in K_VALUES:
        print(f"\n--- Membuat {k}-Shot ---")
        try:
            k_path = create_k_shot_subset(k, class_to_images, pairs_map, class_names, yaml_data, DEST_DIR)
            stats = analyze_stats(k, k_path, class_names)
            all_stats.extend(stats)
            print(f"    -> Selesai. YAML: {k}-shot.yaml")
        except ValueError as e:
            print(f"    [FAILED] {e}")
            break # Stop jika data tidak cukup
        except Exception as e:
            print(f"    [ERROR] {e}")
            break

    # 5. Report
    if all_stats:
        csv_path = os.path.join(DEST_DIR, "kshot_class_distribution.csv")
        df = pd.DataFrame(all_stats)
        df.to_csv(csv_path, index=False)
        print(f"\n[*] Laporan CSV tersimpan: {csv_path}")

if __name__ == "__main__":
    main()

[*] Membersihkan folder tujuan: ../dataset_collection/roboflow/k-shot/object_detection_healthy_rust
[*] Source: ../dataset_collection/roboflow/object_detection/object_detection_few_shot_healthy_rust
[*] Dest  : ../dataset_collection/roboflow/k-shot/object_detection_healthy_rust
[*] Kelas: ['healthy', 'rust']

[*] Menyalin folder valid dan test (One-time copy)...
    - Berhasil menyalin valid
    - Berhasil menyalin test

[*] Mengindeks 350 gambar training...


Indexing Train Data: 100%|██████████| 350/350 [00:00<00:00, 77655.86it/s]


[*] Jumlah gambar tersedia per kelas:
    - healthy: 175
    - rust: 175

--- Membuat 5-Shot ---
    -> Selesai. YAML: 5-shot.yaml

--- Membuat 10-Shot ---
    -> Selesai. YAML: 10-shot.yaml

--- Membuat 15-Shot ---
    -> Selesai. YAML: 15-shot.yaml

--- Membuat 20-Shot ---
    -> Selesai. YAML: 20-shot.yaml

--- Membuat 25-Shot ---
    -> Selesai. YAML: 25-shot.yaml

--- Membuat 30-Shot ---
    -> Selesai. YAML: 30-shot.yaml

--- Membuat 35-Shot ---
    -> Selesai. YAML: 35-shot.yaml

--- Membuat 40-Shot ---
    -> Selesai. YAML: 40-shot.yaml

--- Membuat 45-Shot ---
    -> Selesai. YAML: 45-shot.yaml

--- Membuat 50-Shot ---
    -> Selesai. YAML: 50-shot.yaml

[*] Laporan CSV tersimpan: ../dataset_collection/roboflow/k-shot/object_detection_healthy_rust\kshot_class_distribution.csv
